# SCoRE Package Demonstration
This notebook demonstrates how to import and use the functionality provided by the `SCoRE` package for data generation, model fitting, and risk-controlled selection.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Import from our newly created SCoRE package
from SCoRE import gen_data_Jin2023, loss_Jin2023
from SCoRE import SCoRE_MDR, eval_MDR

### 1. Generate Synthetic Data
We use `gen_data_Jin2023` to generate synthetic data and split it into training, calibration, and test sets.

In [2]:
# Setup data parameters
n_samples = 2000
sig = 1.0
dim = 20

# Generate data
X, mu_x, eps, Y = gen_data_Jin2023(setting=1, n=n_samples, sig=sig, dim=dim)

# Split datasets
X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.6, random_state=42)
X_calib, X_test, Y_calib, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

print(f"Train size: {len(X_train)}, Calib size: {len(X_calib)}, Test size: {len(X_test)}")

Train size: 800, Calib size: 600, Test size: 600


### 2. Train a Model
Train a simple model on the training set and use it to obtain predictions on the calibration and test sets.

In [3]:
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, Y_train)

# Get predictions
Y_calib_pred = model.predict(X_calib)
Y_test_pred = model.predict(X_test)

### 3. Compute Losses
Compute the true loss on the calibration set (`Lcalib`) and the predicted loss on both calibration (`Lcalib_pred`) and test sets (`Ltest_pred`). 

In [4]:
import warnings
warnings.filterwarnings('ignore')

# Using tau=np.inf for 0-1 loss (y <= 0)
tau = np.inf
Lcalib = loss_Jin2023(Y_calib, tau)

# Using the same loss function on predicted Y as a proxy for predicted loss
Lcalib_pred = loss_Jin2023(Y_calib_pred, tau)
Ltest_pred = loss_Jin2023(Y_test_pred, tau)

# Define Dcalib and Dtest tuples
Dcalib = (Lcalib, Lcalib_pred)
Dtest = (np.zeros_like(Ltest_pred), Ltest_pred)  # True Ltest should not be accessed by SCoRE

### 4. Apply SCoRE Risk Control
Run `SCoRE_MDR` to select a subset of the test instances while controlling the risk.

In [5]:
alpha = 0.3
gamma = 0.3

selected_indices = SCoRE_MDR(Dcalib, Dtest, alpha=alpha, gamma=gamma)
print(f"Selected {len(selected_indices)} out of {len(Ltest_pred)} test instances.")

Selected 281 out of 600 test instances.


### 5. Evaluate the Selection
Evaluate the empirical risk and reward of the selected instances using `eval_MDR`.

In [ ]:
# Compute true Ltest just for evaluation purposes
Ltest = loss_Jin2023(Y_test, tau)

# Assume a random variable for reward for the sake of the example
Rtest = np.random.uniform(0, 1, size=len(Ltest_pred))

# Evaluate
risk_acc, reward_acc = eval_MDR(Ltest, Rtest, selected_indices)
print(f"Empirical Risk on selected (should be <= alpha ideally): {risk_acc:.4f}")
print(f"Total Reward of selected: {reward_acc:.2f}")

Empirical Risk on selected (should be <= beta ideally): 0.2200
Total Reward of selected: 137.52
